# Imports

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets


from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


batch_size = 64
num_classes = 9
learning_rate = 0.01
num_epochs = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transforms

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomRotation((0,180)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(
        mean = [0.5,0.5,0.5],
        std = [0.5,0.5,0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.5,0.5,0.5],
        std = [0.5,0.5,0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.5,0.5,0.5],
        std = [0.5,0.5,0.5]
    )
])

# Datasets

In [3]:
train_dataset = datasets.ImageFolder(
    root = 'C:/Users/Robyn/Desktop/Research Project/cards_split/train',
    transform = train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 4,
)

test_dataset = datasets.ImageFolder(
    root = 'C:/Users/Robyn/Desktop/Research Project/cards_split/test',
    transform = test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 4,
)

val_dataset = datasets.ImageFolder(
    root = 'C:/Users/Robyn/Desktop/Research Project/cards_split/val',
    transform = val_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 4,
)

# CNN Model
### Code adapted from: https://www.digitalocean.com/community/tutorials/writing-cnns-from-scratch-in-pytorch

In [4]:
class cnn_model(nn.Module):
    def __init__(self, num_classes):
        super(cnn_model, self).__init__()
        self.conv_layer1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv_layer2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_layer3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv_layer4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.pool = nn.AdaptiveMaxPool2d((7,7))

        self.fc1 = nn.Linear(3136,128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self,x):
        out = self.conv_layer1(x)
        out = self.conv_layer2(out)
        out = self.max_pool1(out)

        out = self.conv_layer3(out)
        out = self.conv_layer4(out)
        out = self.max_pool2(out)

        out = self.pool(out)

        out = out.reshape(out.size(0), -1)

        out = self.fc1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        return out

# Hyperparameters
### Code adapted from: https://www.digitalocean.com/community/tutorials/writing-cnns-from-scratch-in-pytorch

In [5]:
model = cnn_model(num_classes)

# Loss Function
criterion = nn.CrossEntropyLoss()

#Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)

total_step = len(train_loader)

# Training
### Code adapted from: https://www.digitalocean.com/community/tutorials/writing-cnns-from-scratch-in-pytorch

In [6]:
model.train()

for epoch in range(num_epochs):
    # Load data batches with dataloader
    for i,(images,labels) in enumerate(train_loader):
        # Move tensors to GPU or CPU
        images = images.to(device)
        labels = labels.to(device)

        #Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        #Backward Propagation and Optimisation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print('Epoch [{},{}], Loss: {:.4f}'.format(epoch+1, num_epochs, loss.item()))

Epoch [1,20], Loss: 0.2932
Epoch [2,20], Loss: 0.0354
Epoch [3,20], Loss: 0.0315
Epoch [4,20], Loss: 0.0154
Epoch [5,20], Loss: 0.6058
Epoch [6,20], Loss: 0.0673
Epoch [7,20], Loss: 0.0244
Epoch [8,20], Loss: 0.0132
Epoch [9,20], Loss: 0.0040
Epoch [10,20], Loss: 0.0053
Epoch [11,20], Loss: 0.0016
Epoch [12,20], Loss: 0.0031
Epoch [13,20], Loss: 0.0040
Epoch [14,20], Loss: 0.0007
Epoch [15,20], Loss: 0.0042
Epoch [16,20], Loss: 0.0046


KeyboardInterrupt: 

# Testing

In [8]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0

    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        output = model(images)
        _, predicted = torch.max(output.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy: {}%'.format(100 * correct / total))


Accuracy: 99.70824215900802%
